In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
import os, csv, json, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter, defaultdict
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
 
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
 
annotations = {}
with open(os.path.join(ZIP1, "annotation.csv"), 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            cid = row[0].strip()
            annotations[cid] = {
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
            }
 
samples = []
with open(os.path.join(ZIP1, "sample.csv"), 'r') as f:
    reader = csv.reader(f); next(reader)
    for row in reader: samples.append(row)
 
emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5,'surprise':6}
polarity_map = {'positive':0,'neutral':1,'negative':2}
intensity_map = {'weak':0,'powerful':1}
emo_names = ['angry','disgust','fear','happy','neutral','sad','surprise']
 
mcis_index = []
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {'sample_id': s[0].strip(), 'clip_ids': clips,
             'feature_files': [c.replace('/','_')+'.pt' for c in clips]}
    for ln, ci in [('clip3',2),('clip4',3)]:
        cid = clips[ci]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{ln}_emotion'] = emotion_map.get(ann['emotion'],-1)
            entry[f'{ln}_polarity'] = polarity_map.get(ann.get('polarity',''),-1)
            entry[f'{ln}_intensity'] = intensity_map.get(ann.get('intensity',''),-1)
        else:
            entry[f'{ln}_emotion'] = -1
            entry[f'{ln}_polarity'] = -1
            entry[f'{ln}_intensity'] = -1
    mcis_index.append(entry)
 
def split_mcis(mcis_index, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    clip4_groups = {}
    for idx, e in enumerate(mcis_index):
        c4 = e['clip_ids'][3]
        clip4_groups.setdefault(c4, []).append(idx)
    keys = list(clip4_groups.keys()); rng.shuffle(keys)
    n = len(keys); nt = int(n*train_ratio); nv = int(n*val_ratio)
    return ([i for g in keys[:nt] for i in clip4_groups[g]],
            [i for g in keys[nt:nt+nv] for i in clip4_groups[g]],
            [i for g in keys[nt+nv:] for i in clip4_groups[g]])
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Device: cuda
Train: 1980, Val: 424, Test: 426


In [5]:
class HiEFDataset(Dataset):
    def __init__(self, mcis_index, features_dir, indices):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
    def __len__(self): return len(self.indices)
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        c3f,c3o,c3t,c3a = self._load(e['feature_files'][2])
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            'clip3_face':c3f,'clip3_ori':c3o,'clip3_text':c3t,'clip3_audio':c3a,
            'target':e['clip4_emotion'],
            'clip3_emotion':e['clip3_emotion'],
            'clip4_polarity':e['clip4_polarity'],
            'clip4_intensity':e['clip4_intensity'],
        }
 
def collate_fn(batch):
    r = {}
    for k in [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]:
        r[k] = torch.stack([b[k] for b in batch])
    for k in ['target','clip3_emotion','clip4_polarity','clip4_intensity']:
        r[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
    return r
 
BATCH_SIZE = 32
train_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print("Data loaded ✓")
 
# %%
class TemporalTransformer(nn.Module):
    def __init__(self, d=512, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, 16, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, n_heads, d*4, dropout, batch_first=True, norm_first=True)
        self.t = nn.TransformerEncoder(el, num_layers=n_layers)
    def forward(self, x): return self.t(x + self.pos[:, :x.size(1), :])
 
class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, n_heads=8, n_layers=1, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
    def forward(self, q, kv):
        x = q
        for attn, norm in zip(self.layers, self.norms):
            o, _ = attn(x, kv, kv); x = norm(x + o)
        return x
 
class ClipEncoder(nn.Module):
    """Encode a single clip → 512-d feature."""
    def __init__(self, d=512):
        super().__init__()
        self.face_t = TemporalTransformer(d)
        self.ori_t = TemporalTransformer(d)
        self.type_f = CrossAttentionFusion(d)
        self.audio_proj = nn.Linear(527, d)
        self.mod_f = CrossAttentionFusion(d)
    def forward(self, face, ori, text, audio):
        f = self.face_t(face).mean(1, keepdim=True)
        o = self.ori_t(ori).mean(1, keepdim=True)
        v = self.type_f(f, torch.cat([f, o], 1))
        a = self.audio_proj(F.normalize(audio, dim=-1)).unsqueeze(1)
        t = text.unsqueeze(1)
        return self.mod_f(v, torch.cat([v, t, a], 1)).squeeze(1)
 
def make_head(d=512, n_classes=7):
    return nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d//2), 
                         nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n_classes))

Data loaded ✓


In [6]:
class M1_ContextOnly(nn.Module):
    """P(B|C): clips I+II only, no clip III."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.ctx_fusion = CrossAttentionFusion(d)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        ctx = torch.stack([c1, c2], dim=1)
        fused = self.ctx_fusion(c1.unsqueeze(1), ctx).squeeze(1)
        return self.head(fused)
 
 
class M2_AOnly(nn.Module):
    """P(B|A): clip III only, no context."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.head = make_head(d)
    def forward(self, batch):
        a = self.enc(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        return self.head(a)
 
 
class M3_Full(nn.Module):
    """P(B|C,A): clips I+II+III, LSTM+Transformer."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.lstm = nn.LSTM(d, d, num_layers=3, batch_first=False, dropout=0.1)
        self.pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, 8, d*4, 0.1, batch_first=True, norm_first=True)
        self.trans = nn.TransformerEncoder(el, num_layers=2)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        c3 = self.enc(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        seq = torch.stack([c1, c2, c3], dim=0)
        out, _ = self.lstm(seq)
        out = self.trans(out.permute(1,0,2) + self.pos).mean(1)
        return self.head(out)

In [7]:
def compute_metrics(preds, labels, n_classes=7):
    preds, labels = np.array(preds), np.array(labels)
    war = (preds == labels).sum() / len(labels) * 100
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0: recalls.append((preds[mask] == c).sum() / mask.sum() * 100)
    return war, np.mean(recalls) if recalls else 0.0
 
def train_and_evaluate(model, train_loader, val_loader, test_loader, 
                       n_epochs=30, lr=1e-4, name="model", device=DEVICE):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    best_uar = 0; best_epoch = 0
    
    for epoch in range(1, n_epochs+1):
        model.train()
        for batch in train_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            loss = F.cross_entropy(model(bg), bg['target'])
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        
        model.eval(); vp, vl = [], []; vl_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                bg = {k: v.to(device) for k, v in batch.items()}
                lo = model(bg)
                vl_loss += F.cross_entropy(lo, bg['target']).item() * lo.size(0)
                vp.extend(lo.argmax(-1).cpu().numpy()); vl.extend(batch['target'].numpy())
        scheduler.step(vl_loss / len(val_loader.dataset))
        _, vu = compute_metrics(vp, vl)
        if vu > best_uar: best_uar = vu; best_epoch = epoch; torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
        if epoch % 10 == 0 or epoch == 1:
            vw, _ = compute_metrics(vp, vl)
            print(f"  [{name}] Ep {epoch:>3}: Val WAR={vw:.1f}%, UAR={vu:.1f}%{'  ★' if epoch==best_epoch else ''}")
    
    # Test with best checkpoint
    model.load_state_dict(torch.load(f'/kaggle/working/{name}.pt', map_location=device, weights_only=True))
    model.eval(); tp, tl, t_pol, t_int = [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            bg = {k: v.to(device) for k, v in batch.items()}
            tp.extend(model(bg).argmax(-1).cpu().numpy())
            tl.extend(batch['target'].numpy())
            t_pol.extend(batch['clip4_polarity'].numpy())
            t_int.extend(batch['clip4_intensity'].numpy())
    
    tw, tu = compute_metrics(tp, tl)
    return {
        'name': name, 'test_war': tw, 'test_uar': tu, 'best_epoch': best_epoch,
        'preds': np.array(tp), 'labels': np.array(tl),
        'polarity': np.array(t_pol), 'intensity': np.array(t_int),
    }

In [8]:
print("=" * 70)
print("CONTRIBUTION DECOMPOSITION: What drives B's emotion?")
print("=" * 70)
 
results = {}
 
# --- M0: Majority class baseline ---
print("\n[M0] P(B) — majority class baseline")
train_labels = [mcis_index[i]['clip4_emotion'] for i in train_idx]
majority_class = Counter(train_labels).most_common(1)[0][0]
print(f"  Majority class: {emo_names[majority_class]}")
 
test_labels_m0 = np.array([mcis_index[i]['clip4_emotion'] for i in test_idx])
test_pols_m0 = np.array([mcis_index[i]['clip4_polarity'] for i in test_idx])
test_ints_m0 = np.array([mcis_index[i]['clip4_intensity'] for i in test_idx])
m0_preds = np.full_like(test_labels_m0, majority_class)
m0_war, m0_uar = compute_metrics(m0_preds, test_labels_m0)
print(f"  Test WAR={m0_war:.1f}%, UAR={m0_uar:.1f}%")
results['M0'] = {
    'name': 'M0_majority', 'test_war': m0_war, 'test_uar': m0_uar, 'best_epoch': 0,
    'preds': m0_preds, 'labels': test_labels_m0,
    'polarity': test_pols_m0, 'intensity': test_ints_m0,
}
 
# --- M1: Context only ---
print("\n[M1] P(B|C) — context only (clips I+II)")
m = M1_ContextOnly().to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
results['M1'] = train_and_evaluate(m, train_loader, val_loader, test_loader, n_epochs=30, name="M1_context")
 
# --- M2: A only ---
print("\n[M2] P(B|A) — A only (clip III)")
m = M2_AOnly().to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
results['M2'] = train_and_evaluate(m, train_loader, val_loader, test_loader, n_epochs=30, name="M2_A_only")
 
# --- M3: Full ---
print("\n[M3] P(B|C,A) — full (clips I+II+III)")
m = M3_Full().to(DEVICE)
print(f"  Parameters: {sum(p.numel() for p in m.parameters()):,}")
results['M3'] = train_and_evaluate(m, train_loader, val_loader, test_loader, n_epochs=30, name="M3_full")

CONTRIBUTION DECOMPOSITION: What drives B's emotion?

[M0] P(B) — majority class baseline
  Majority class: happy
  Test WAR=23.2%, UAR=14.3%

[M1] P(B|C) — context only (clips I+II)


/tmp/ipykernel_58/1316552975.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=n_layers)


  Parameters: 16,185,351
  [M1_context] Ep   1: Val WAR=24.5%, UAR=15.1%  ★
  [M1_context] Ep  10: Val WAR=32.1%, UAR=22.6%
  [M1_context] Ep  20: Val WAR=32.3%, UAR=23.0%
  [M1_context] Ep  30: Val WAR=31.8%, UAR=23.0%

[M2] P(B|A) — A only (clip III)
  Parameters: 15,133,703


/tmp/ipykernel_58/1316552975.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=n_layers)


  [M2_A_only] Ep   1: Val WAR=30.4%, UAR=18.5%  ★
  [M2_A_only] Ep  10: Val WAR=31.6%, UAR=22.3%
  [M2_A_only] Ep  20: Val WAR=28.5%, UAR=21.9%
  [M2_A_only] Ep  30: Val WAR=32.3%, UAR=23.9%

[M3] P(B|C,A) — full (clips I+II+III)
  Parameters: 27,743,751


/tmp/ipykernel_58/4008224265.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  [M3_full] Ep   1: Val WAR=23.1%, UAR=14.3%  ★
  [M3_full] Ep  10: Val WAR=37.5%, UAR=24.9%  ★
  [M3_full] Ep  20: Val WAR=41.0%, UAR=28.4%  ★
  [M3_full] Ep  30: Val WAR=35.4%, UAR=24.4%


In [9]:
print("\n" + "=" * 70)
print("CONTRIBUTION ANALYSIS")
print("=" * 70)
 
# Overall results
print(f"\n{'Model':<25} | {'Input':>20} | {'Test WAR':>8} | {'Test UAR':>8}")
print("-" * 70)
for key in ['M0', 'M1', 'M2', 'M3']:
    r = results[key]
    inputs = {'M0':'nothing (majority)', 'M1':'clips I+II (context)', 'M2':'clip III (A only)', 'M3':'clips I+II+III (full)'}
    print(f"{r['name']:<25} | {inputs[key]:>20} | {r['test_war']:>7.1f}% | {r['test_uar']:>7.1f}%")
 
# Deltas
print(f"\n--- Incremental contributions ---")
delta_a = results['M3']['test_uar'] - results['M1']['test_uar']
delta_c = results['M3']['test_uar'] - results['M2']['test_uar']
delta_over_majority = results['M3']['test_uar'] - results['M0']['test_uar']
 
print(f"  Δ_A = M3 - M1 = {delta_a:+.1f} pts  (incremental value of A given context)")
print(f"  Δ_C = M3 - M2 = {delta_c:+.1f} pts  (incremental value of context given A)")
print(f"  Δ_total = M3 - M0 = {delta_over_majority:+.1f} pts  (total model value over random)")
print(f"  M1 - M0 = {results['M1']['test_uar'] - results['M0']['test_uar']:+.1f} pts  (context alone over majority)")
print(f"  M2 - M0 = {results['M2']['test_uar'] - results['M0']['test_uar']:+.1f} pts  (A alone over majority)")


CONTRIBUTION ANALYSIS

Model                     |                Input | Test WAR | Test UAR
----------------------------------------------------------------------
M0_majority               |   nothing (majority) |    23.2% |    14.3%
M1_context                | clips I+II (context) |    36.4% |    23.1%
M2_A_only                 |    clip III (A only) |    36.6% |    25.0%
M3_full                   | clips I+II+III (full) |    38.0% |    26.4%

--- Incremental contributions ---
  Δ_A = M3 - M1 = +3.2 pts  (incremental value of A given context)
  Δ_C = M3 - M2 = +1.3 pts  (incremental value of context given A)
  Δ_total = M3 - M0 = +12.1 pts  (total model value over random)
  M1 - M0 = +8.8 pts  (context alone over majority)
  M2 - M0 = +10.7 pts  (A alone over majority)


In [10]:
print("\n" + "=" * 70)
print("Δ_A BY INTERACTION TYPE")
print("=" * 70)
print("(Δ_A = how much does knowing A's emotion help, given context?)")
 
pol_names = {0:'Positive', 1:'Neutral', 2:'Negative'}
int_names = {0:'Weak', 1:'Powerful'}
 
# Compute per-slice UAR for M1 and M3
def slice_uar(preds, labels, mask):
    if mask.sum() < 5: return None
    return compute_metrics(preds[mask], labels[mask])[1]
 
print(f"\n{'Interaction type':<25} | {'M1 UAR':>7} | {'M3 UAR':>7} | {'Δ_A':>7} | {'n':>5}")
print("-" * 60)
 
delta_a_slices = {}
for pol in [0, 1, 2]:
    for inten in [0, 1]:
        mask = (results['M3']['polarity'] == pol) & (results['M3']['intensity'] == inten)
        n = mask.sum()
        if n < 5: continue
        
        uar_m1 = slice_uar(results['M1']['preds'], results['M1']['labels'], mask)
        uar_m3 = slice_uar(results['M3']['preds'], results['M3']['labels'], mask)
        
        if uar_m1 is not None and uar_m3 is not None:
            delta = uar_m3 - uar_m1
            label = f"{pol_names[pol]} + {int_names[inten]}"
            delta_a_slices[label] = {'m1': uar_m1, 'm3': uar_m3, 'delta': delta, 'n': n}
            print(f"  {label:<23} | {uar_m1:>6.1f}% | {uar_m3:>6.1f}% | {delta:>+6.1f} | {n:>5}")
 
# Also slice by polarity only
print(f"\n{'Polarity only':<25} | {'M1 UAR':>7} | {'M3 UAR':>7} | {'Δ_A':>7} | {'n':>5}")
print("-" * 60)
for pol in [0, 1, 2]:
    mask = results['M3']['polarity'] == pol
    n = mask.sum()
    if n < 5: continue
    uar_m1 = slice_uar(results['M1']['preds'], results['M1']['labels'], mask)
    uar_m3 = slice_uar(results['M3']['preds'], results['M3']['labels'], mask)
    if uar_m1 is not None and uar_m3 is not None:
        delta = uar_m3 - uar_m1
        print(f"  {pol_names[pol]:<23} | {uar_m1:>6.1f}% | {uar_m3:>6.1f}% | {delta:>+6.1f} | {n:>5}")
 
# Also slice by intensity only
print(f"\n{'Intensity only':<25} | {'M1 UAR':>7} | {'M3 UAR':>7} | {'Δ_A':>7} | {'n':>5}")
print("-" * 60)
for inten in [0, 1]:
    mask = results['M3']['intensity'] == inten
    n = mask.sum()
    if n < 5: continue
    uar_m1 = slice_uar(results['M1']['preds'], results['M1']['labels'], mask)
    uar_m3 = slice_uar(results['M3']['preds'], results['M3']['labels'], mask)
    if uar_m1 is not None and uar_m3 is not None:
        delta = uar_m3 - uar_m1
        print(f"  {int_names[inten]:<23} | {uar_m1:>6.1f}% | {uar_m3:>6.1f}% | {delta:>+6.1f} | {n:>5}")



Δ_A BY INTERACTION TYPE
(Δ_A = how much does knowing A's emotion help, given context?)

Interaction type          |  M1 UAR |  M3 UAR |     Δ_A |     n
------------------------------------------------------------
  Positive + Weak         |   12.5% |   36.9% |  +24.4 |    68
  Positive + Powerful     |   26.5% |   28.8% |   +2.4 |    41
  Neutral + Weak          |   18.3% |   21.9% |   +3.5 |    85
  Neutral + Powerful      |   38.0% |   32.0% |   -6.0 |    34
  Negative + Weak         |   22.0% |   21.6% |   -0.4 |   141
  Negative + Powerful     |   32.2% |   45.8% |  +13.6 |    57

Polarity only             |  M1 UAR |  M3 UAR |     Δ_A |     n
------------------------------------------------------------
  Positive                |   16.2% |   33.1% |  +16.9 |   109
  Neutral                 |   18.3% |   21.8% |   +3.5 |   119
  Negative                |   22.2% |   23.9% |   +1.7 |   198

Intensity only            |  M1 UAR |  M3 UAR |     Δ_A |     n
----------------------------

In [11]:
print("\n" + "=" * 70)
print("PER-CLASS: Which emotions benefit from knowing A?")
print("=" * 70)
 
print(f"\n{'Emotion':<12} | {'M0':>6} | {'M1(C)':>6} | {'M2(A)':>6} | {'M3(CA)':>6} | {'Δ_A':>6} | {'Δ_C':>6}")
print("-" * 65)
 
for c in range(7):
    mask = results['M3']['labels'] == c
    n = mask.sum()
    if n == 0: continue
    
    r0 = (results['M0']['preds'][mask] == c).sum() / n * 100
    r1 = (results['M1']['preds'][mask] == c).sum() / n * 100
    r2 = (results['M2']['preds'][mask] == c).sum() / n * 100
    r3 = (results['M3']['preds'][mask] == c).sum() / n * 100
    
    da = r3 - r1
    dc = r3 - r2
    
    print(f"  {emo_names[c]:<10} | {r0:>5.1f}% | {r1:>5.1f}% | {r2:>5.1f}% | {r3:>5.1f}% | {da:>+5.1f} | {dc:>+5.1f}  (n={n})")



PER-CLASS: Which emotions benefit from knowing A?

Emotion      |     M0 |  M1(C) |  M2(A) | M3(CA) |    Δ_A |    Δ_C
-----------------------------------------------------------------
  angry      |   0.0% |  27.6% |  57.9% |  40.8% | +13.2 | -17.1  (n=76)
  disgust    |   0.0% |   0.0% |   4.4% |   0.0% |  +0.0 |  -4.4  (n=45)
  fear       |   0.0% |   0.0% |   0.0% |   0.0% |  +0.0 |  +0.0  (n=4)
  happy      | 100.0% |  45.5% |  43.4% |  39.4% |  -6.1 |  -4.0  (n=99)
  neutral    |   0.0% |  75.0% |  54.6% |  63.0% | -12.0 |  +8.3  (n=108)
  sad        |   0.0% |  13.8% |  12.1% |  41.4% | +27.6 | +29.3  (n=58)
  surprise   |   0.0% |   0.0% |   2.8% |   0.0% |  +0.0 |  -2.8  (n=36)


In [12]:
print("\n" + "=" * 70)
print("CRITICAL TEST: Does A actually add value over context alone?")
print("=" * 70)
 
m1_uar = results['M1']['test_uar']
m3_uar = results['M3']['test_uar']
delta = m3_uar - m1_uar
 
print(f"\n  P(B|C)   = M1 UAR: {m1_uar:.1f}%")
print(f"  P(B|C,A) = M3 UAR: {m3_uar:.1f}%")
print(f"  Δ_A      = {delta:+.1f} percentage points")
 
if abs(delta) < 2.0:
    print(f"\n  ⚠ FINDING: A's emotion provides MINIMAL additional signal over context alone.")
    print(f"  This suggests that EF performance is primarily driven by shared context,")
    print(f"  not interpersonal emotional influence as the Hi-EF paper assumes.")
elif delta > 2.0:
    print(f"\n  ✓ A's emotion provides meaningful additional signal over context.")
    print(f"  Interpersonal influence is real, though its strength varies by interaction type.")
else:
    print(f"\n  ✗ A's emotion actually HURTS prediction when added to context.")
    print(f"  This may indicate that clip III introduces noise (identity, camera angle)")
    print(f"  that disrupts context-based prediction.")
 
# Check hypothesis: Δ_A_powerful > Δ_A_weak
print(f"\n--- Hypothesis test: Δ_A(powerful) > Δ_A(weak) ---")
mask_weak = results['M3']['intensity'] == 0
mask_powerful = results['M3']['intensity'] == 1
 
da_weak = slice_uar(results['M3']['preds'], results['M3']['labels'], mask_weak) - \
          slice_uar(results['M1']['preds'], results['M1']['labels'], mask_weak)
da_powerful = slice_uar(results['M3']['preds'], results['M3']['labels'], mask_powerful) - \
              slice_uar(results['M1']['preds'], results['M1']['labels'], mask_powerful)
 
print(f"  Δ_A(weak):     {da_weak:+.1f} pts")
print(f"  Δ_A(powerful): {da_powerful:+.1f} pts")
 
if da_powerful > da_weak:
    print(f"  ✓ CONFIRMED: A is more informative in powerful interactions ({da_powerful:+.1f} > {da_weak:+.1f})")
    print(f"  This matches MI finding: I(E_A;E_B|powerful) > I(E_A;E_B|weak)")
else:
    print(f"  ✗ NOT confirmed: Δ_A does not increase with intensity")



CRITICAL TEST: Does A actually add value over context alone?

  P(B|C)   = M1 UAR: 23.1%
  P(B|C,A) = M3 UAR: 26.4%
  Δ_A      = +3.2 percentage points

  ✓ A's emotion provides meaningful additional signal over context.
  Interpersonal influence is real, though its strength varies by interaction type.

--- Hypothesis test: Δ_A(powerful) > Δ_A(weak) ---
  Δ_A(weak):     +1.2 pts
  Δ_A(powerful): +13.9 pts
  ✓ CONFIRMED: A is more informative in powerful interactions (+13.9 > +1.2)
  This matches MI finding: I(E_A;E_B|powerful) > I(E_A;E_B|weak)


In [13]:
save_data = {
    'overall': {k: {'war': r['test_war'], 'uar': r['test_uar'], 'best_epoch': r.get('best_epoch',0)} 
                for k, r in results.items()},
    'delta_a_global': delta,
    'delta_a_powerful': da_powerful,
    'delta_a_weak': da_weak,
    'delta_a_slices': {k: {'m1': v['m1'], 'm3': v['m3'], 'delta': v['delta'], 'n': int(v['n'])} 
                       for k, v in delta_a_slices.items()},
}
 
with open('/kaggle/working/contribution_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)
 
print("Saved to /kaggle/working/contribution_results.json")


Saved to /kaggle/working/contribution_results.json
